# Day 33: Semantic Caching for LLM Cost Optimisation

Implement a cache that returns a cached response for semantically similar queries.

In [ ]:
import os
import hashlib
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. Exact‑match cache (dictionary)

In [ ]:
exact_cache = {}

def exact_cached_completion(prompt: str) -> str:
    if prompt in exact_cache:
        print("Cache hit (exact)")
        return exact_cache[prompt]
    print("Calling OpenAI...")
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )
    answer = response.choices[0].message.content
    exact_cache[prompt] = answer
    return answer

print(exact_cached_completion("What is AI?"))
print(exact_cached_completion("What is AI?"))  # second time hits cache

## 2. Semantic cache with FAISS (in‑memory)
Store query embeddings and responses; retrieve by cosine similarity.

In [ ]:
import faiss

class SemanticCacheFAISS:
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.embeddings = []
        self.responses = []
        self.index = None
    
    def _embed(self, text: str) -> np.ndarray:
        resp = client.embeddings.create(model="text-embedding-3-small", input=[text])
        return np.array(resp.data[0].embedding).astype('float32')
    
    def get(self, query: str):
        if not self.index:
            return None
        q_emb = self._embed(query).reshape(1, -1)
        faiss.normalize_L2(q_emb)
        scores, indices = self.index.search(q_emb, 1)
        if scores[0][0] >= self.threshold:
            return self.responses[indices[0][0]]
        return None
    
    def set(self, query: str, response: str):
        emb = self._embed(query)
        self.embeddings.append(emb)
        self.responses.append(response)
        self._rebuild_index()
    
    def _rebuild_index(self):
        arr = np.vstack(self.embeddings).astype('float32')
        faiss.normalize_L2(arr)
        self.index = faiss.IndexFlatIP(arr.shape[1])
        self.index.add(arr)

semantic_cache = SemanticCacheFAISS(threshold=0.9)

def cached_completion(prompt: str) -> str:
    cached = semantic_cache.get(prompt)
    if cached:
        print("Semantic cache hit!")
        return cached
    print("Calling OpenAI...")
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )
    answer = response.choices[0].message.content
    semantic_cache.set(prompt, answer)
    return answer

print(cached_completion("Explain quantum computing."))
print(cached_completion("What is quantum computing?"))  # semantically similar, should hit cache

## 3. Redis with vector search (optional, requires Redis Stack)
We'll use `redisvl` to create an index and store vectors.

In [ ]:
# Uncomment if Redis is running locally
# from redisvl import RedisVL
# from redisvl.schema import IndexSchema
# 
# schema = IndexSchema.from_dict({
#     "index": {"name": "llm_cache"},
#     "fields": [
#         {"name": "query", "type": "text"},
#         {"name": "response", "type": "text"},
#         {"name": "embedding", "type": "vector", "attrs": {"dims": 1536, "algorithm": "flat"}}
#     ]
# })
# rvl = RedisVL(schema)
# rvl.create_index()
print("Redis setup requires Redis Stack – see exercise 3 for details.")

## 4. Cache with TTL and size limits

In [ ]:
import time
from collections import OrderedDict

class TTLCache:
    def __init__(self, ttl_seconds=3600, max_size=100):
        self.ttl = ttl_seconds
        self.max_size = max_size
        self.cache = OrderedDict()  # key -> (value, timestamp)
    
    def get(self, key):
        if key not in self.cache:
            return None
        value, timestamp = self.cache[key]
        if time.time() - timestamp > self.ttl:
            del self.cache[key]
            return None
        self.cache.move_to_end(key)
        return value
    
    def set(self, key, value):
        if len(self.cache) >= self.max_size:
            self.cache.popitem(last=False)
        self.cache[key] = (value, time.time())

# Usage: wrap around exact or semantic cache